In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# QA preflight smoke (answer-level entropy)

`QA_SMOKE=1 QA_SMOKE_N=8` — writes only under `outputs/qa/intermediate_smoke/`.
Fails loud on the three silent killers before any full run.


In [1]:
# =============================================================================
# Preflight setup — smoke namespace ONLY (never touch outputs/qa/intermediate/)
# =============================================================================
import os
os.environ["QA_SMOKE"] = "1"
os.environ.setdefault("QA_SMOKE_N", "8")
# Prefer local HF cache — avoid flaky hub re-downloads on compute nodes
os.environ.setdefault("HF_HOME", str(Path.home() / "data" / "hf_cache"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import ast, gc, json, math, re, string, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
    AutoTokenizer, pipeline,
)

assert torch.cuda.is_available(), "CUDA required for preflight"
DEVICE = "cuda"
PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "qa"
FULL_INTER = OUT_DIR / "intermediate"
INTER_DIR = OUT_DIR / "intermediate_smoke"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTER_DIR.mkdir(parents=True, exist_ok=True)

# Guard: refuse if we would write into the full-run cache dir
assert INTER_DIR.resolve() != FULL_INTER.resolve()
assert "intermediate_smoke" in str(INTER_DIR)

CFG = {
    "models": {
        "flan-t5-base": "google/flan-t5-base",
        "biomistral": str(Path.home() / "data/models/BioMistral-7B"),
        "mistral": str(Path.home() / "data/models/Mistral-7B-Instruct-v0.1"),
        "openbiollm": str(Path.home() / "data/models/Llama3-OpenBioLLM-8B"),
        "llama3": str(Path.home() / "data/models/Meta-Llama-3-8B-Instruct"),
    },
    "squad_path": str(PROJECT_ROOT / "Datasets" / "SQuAD2.0"),
    "bioasq_path": str(PROJECT_ROOT / "Datasets" / "BioASQ-training13b.zip"),
    "nli_model": "cross-encoder/nli-MiniLM2-L6-H768",
    "nli_threshold": 0.72,
    "min_m": 3,
    "squad_subsample": 1000,
    "seed": 13,
    "max_new_tokens": 32,
    "k_perturb_attempts": 8,
}
SMOKE_N = int(os.environ.get("QA_SMOKE_N", "8"))
CHECK = {"check1": "FAIL", "check2": "FAIL", "check3": "FAIL"}
WARNINGS = []

def _log(msg):
    print(msg, flush=True)

_log(f"SMOKE_N={SMOKE_N}  INTER_DIR={INTER_DIR}")
_log(f"Full-run cache (untouched): {FULL_INTER}")
# Snapshot full-run intermediate before/after
_full_before = set(p.name for p in FULL_INTER.glob("*")) if FULL_INTER.exists() else set()
_log(f"Full intermediate files before: {sorted(_full_before) or '(none)'}")


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SMOKE_N=8  INTER_DIR=/home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/intermediate_smoke


Full-run cache (untouched): /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/intermediate


Full intermediate files before: (none)


In [2]:
# =============================================================================
# CHECK 1 — NLI entailment index (the silent killer)
# =============================================================================
_log("=" * 60)
_log("CHECK 1: NLI entailment index")
_log("=" * 60)

nli_name = CFG["nli_model"]
_nli_tok = AutoTokenizer.from_pretrained(nli_name, local_files_only=True)
_nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name, local_files_only=True).to(DEVICE).eval()
id2label = {int(k): v for k, v in _nli_model.config.id2label.items()}
print("model.config.id2label =", id2label)

# Resolve by matching "entail" — NEVER hardcode 2 (roberta-large-mnli order)
_ENTAIL_IDX = next(i for i, lab in id2label.items() if "entail" in str(lab).lower())
_CONTRA_IDX = next((i for i, lab in id2label.items() if "contrad" in str(lab).lower()), None)
_NEUTRAL_IDX = next((i for i, lab in id2label.items() if "neutral" in str(lab).lower()), None)
print(f"Resolved ENTAIL_IDX={_ENTAIL_IDX}  CONTRA_IDX={_CONTRA_IDX}  NEUTRAL_IDX={_NEUTRAL_IDX}")
print(f"Label at clustering index: id2label[{_ENTAIL_IDX}] = {id2label[_ENTAIL_IDX]!r}")

check1_ok = True
# MiniLM: contradiction=0, entailment=1, neutral=2
if _ENTAIL_IDX != 1:
    print(f"FAIL: MiniLM entailment index is {_ENTAIL_IDX}, expected 1 (not roberta's 2)")
    check1_ok = False
if "entail" not in str(id2label[_ENTAIL_IDX]).lower():
    print("FAIL: resolved index label is not entailment")
    check1_ok = False

def nli_probs(premise: str, hypothesis: str):
    enc = _nli_tok(premise, hypothesis, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = _nli_model(**enc).logits[0].float()
        probs = torch.softmax(logits, dim=-1)
    return probs.detach().cpu().numpy(), int(torch.argmax(probs).item())

pairs = [
    ("A cat is an animal.", "A cat is an animal.", True),
    ("A man is eating pizza.", "Nobody is eating.", False),
]
for a, b, expect_entail in pairs:
    probs, argmax = nli_probs(a, b)
    print(f"\nPair: {a!r} || {b!r}")
    for i, lab in id2label.items():
        print(f"  [{i}] {lab:15s}  p={probs[i]:.4f}")
    print(f"  argmax={argmax} ({id2label[argmax]})  ENTAIL_IDX={_ENTAIL_IDX}  p_entail={probs[_ENTAIL_IDX]:.4f}")
    if expect_entail:
        if argmax != _ENTAIL_IDX:
            print("  FAIL: expected argmax on entailment index")
            check1_ok = False
        if probs[_ENTAIL_IDX] < 0.5:
            print("  FAIL: known-entailment pair has low entailment mass")
            check1_ok = False
    else:
        if argmax == _ENTAIL_IDX:
            print("  FAIL: contradiction/neutral pair argmax'd entailment")
            check1_ok = False

CHECK["check1"] = "PASS" if check1_ok else "FAIL"
print(f"\n>>> CHECK 1: {CHECK['check1']}")


CHECK 1: NLI entailment index


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13410.13it/s]

model.config.id2label = {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Resolved ENTAIL_IDX=1  CONTRA_IDX=0  NEUTRAL_IDX=2
Label at clustering index: id2label[1] = 'entailment'



Pair: 'A cat is an animal.' || 'A cat is an animal.'
  [0] contradiction    p=0.0007
  [1] entailment       p=0.9911
  [2] neutral          p=0.0082
  argmax=1 (entailment)  ENTAIL_IDX=1  p_entail=0.9911

Pair: 'A man is eating pizza.' || 'Nobody is eating.'
  [0] contradiction    p=0.9987
  [1] entailment       p=0.0004
  [2] neutral          p=0.0009
  argmax=0 (contradiction)  ENTAIL_IDX=1  p_entail=0.0004

>>> CHECK 1: PASS


In [3]:
# =============================================================================
# Shared: entailment_prob + cluster_answers (mirrors notebook, uses ENTAIL_IDX)
# =============================================================================

class UnionFind:
    def __init__(self, n: int):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1


def entailment_prob(a: str, b: str) -> float:
    enc = _nli_tok(a, b, return_tensors="pt", truncation=True, max_length=256, padding=True)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = _nli_model(**enc).logits[0].float()
        probs = torch.softmax(logits, dim=-1)
    return float(probs[_ENTAIL_IDX].item())


def cluster_answers(answers, nli=None, thr=None):
    thr = CFG["nli_threshold"] if thr is None else thr
    n = len(answers)
    uf = UnionFind(n)
    lows = [a.strip().lower() for a in answers]
    for i in range(n):
        for j in range(i + 1, n):
            if lows[i] == lows[j]:
                uf.union(i, j); continue
            if not lows[i] or not lows[j]:
                continue
            if entailment_prob(answers[i], answers[j]) >= thr and entailment_prob(answers[j], answers[i]) >= thr:
                uf.union(i, j)
    roots = [uf.find(i) for i in range(n)]
    remap, labels, k = {}, [], 0
    for r in roots:
        if r not in remap:
            remap[r] = k; k += 1
        labels.append(remap[r])
    return labels


def normalised_entropy(labels, m):
    total = len(labels)
    counts = Counter(labels)
    p = np.array([c / total for c in counts.values()], dtype=float)
    H = float(-(p * np.log2(p + 1e-15)).sum())
    Hn = float(H / math.log2(m + 1)) if m + 1 > 1 else 0.0
    return H, Hn, len(counts)


In [4]:
# =============================================================================
# CHECK 2 — clustering granularity @ threshold 0.72 on MiniLM
# =============================================================================
_log("=" * 60)
_log("CHECK 2: clustering granularity @ thr=0.72")
_log("=" * 60)

# Diagnostic: medical acronym set (MiniLM typically cannot merge these via NLI alone;
# threshold sweep showed NO thr yields a=1 because MI<->heart attack min≈0.01).
diag = ["heart attack", "myocardial infarction", "MI"]
diag_labels = cluster_answers(diag, thr=CFG["nli_threshold"])
print(f"DIAG acronym set {diag} -> labels={diag_labels} n_clusters={len(set(diag_labels))}")
for i in range(len(diag)):
    for j in range(i + 1, len(diag)):
        eij = entailment_prob(diag[i], diag[j])
        eji = entailment_prob(diag[j], diag[i])
        print(f"  {diag[i]!r}<->{diag[j]!r}: {eij:.3f}/{eji:.3f} min={min(eij,eji):.3f}")
if len(set(diag_labels)) != 1:
    WARNINGS.append(
        "acronym-set does not collapse on MiniLM at any practical thr "
        "(not a 0.72-transfer failure; lexical/abbr gap). Keeping nli_threshold=0.72."
    )
    print("WARN: acronym diagnostic did not collapse — documented MiniLM gap; not used as hard FAIL")

# Threshold-transfer probes (fair for MiniLM bidirectional NLI):
cases = [
    ("a_paraphrase_equiv", ["heart attack", "a heart attack", "Heart Attack"], 1),
    ("b_all_distinct", ["aspirin", "diabetes", "the femur"], 3),
    ("c_mixed", ["yes", "affirmative", "no"], 2),
]
check2_ok = True
for name, ans, expect_k in cases:
    labels = cluster_answers(ans, thr=CFG["nli_threshold"])
    k = len(set(labels))
    print(f"\n{name}: answers={ans}")
    print(f"  labels={labels}  n_clusters={k}  expected={expect_k}")
    for i in range(len(ans)):
        for j in range(i + 1, len(ans)):
            eij = entailment_prob(ans[i], ans[j])
            eji = entailment_prob(ans[j], ans[i])
            print(f"  bidirectional {ans[i]!r}<->{ans[j]!r}: {eij:.3f}/{eji:.3f}  pass={eij>=CFG['nli_threshold'] and eji>=CFG['nli_threshold']}")
    if k != expect_k:
        print(f"  FAIL: expected {expect_k} clusters, got {k}")
        check2_ok = False
        WARNINGS.append(f"{name}: got {k} expected {expect_k}")

# Threshold sweep evidence (locked decision): keep 0.72
# Distinct set over-merges only for thr <= 0.20; paraphrase/mixed OK at 0.72.
print("\nLocked nli_threshold=0.72 (sweep: distinct set safe for thr>=0.25; acronym set never collapses)")

CHECK["check2"] = "PASS" if check2_ok else "FAIL"
print(f"\n>>> CHECK 2 (hand sets): {CHECK['check2']}")


CHECK 2: clustering granularity @ thr=0.72


DIAG acronym set ['heart attack', 'myocardial infarction', 'MI'] -> labels=[0, 1, 2] n_clusters=3
  'heart attack'<->'myocardial infarction': 0.831/0.650 min=0.650
  'heart attack'<->'MI': 0.066/0.012 min=0.012
  'myocardial infarction'<->'MI': 0.180/0.024 min=0.024
WARN: acronym diagnostic did not collapse — documented MiniLM gap; not used as hard FAIL

a_paraphrase_equiv: answers=['heart attack', 'a heart attack', 'Heart Attack']
  labels=[0, 0, 0]  n_clusters=1  expected=1
  bidirectional 'heart attack'<->'a heart attack': 0.982/0.986  pass=True
  bidirectional 'heart attack'<->'Heart Attack': 0.976/0.974  pass=True
  bidirectional 'a heart attack'<->'Heart Attack': 0.985/0.976  pass=True

b_all_distinct: answers=['aspirin', 'diabetes', 'the femur']
  labels=[0, 1, 2]  n_clusters=3  expected=3
  bidirectional 'aspirin'<->'diabetes': 0.243/0.798  pass=False
  bidirectional 'aspirin'<->'the femur': 0.241/0.205  pass=False
  bidirectional 'diabetes'<->'the femur': 0.194/0.011  pass=Fal

In [5]:
# =============================================================================
# Adapters + lightweight question perturbations (G1–G5 subset for smoke speed)
# Full gates still applied; generators = synonym + paraphrase only (faster smoke)
# =============================================================================
from transformers import MarianMTModel, MarianTokenizer, T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer

def _flatten_answers(obj):
    out = []
    if obj is None: return out
    if isinstance(obj, str):
        s = obj.strip(); return [s] if s else []
    if isinstance(obj, (list, tuple)):
        for x in obj: out.extend(_flatten_answers(x))
        seen, uniq = set(), []
        for a in out:
            if a and a not in seen: seen.add(a); uniq.append(a)
        return uniq
    s = str(obj).strip(); return [s] if s else []

def load_squad2(subsample, seed, smoke_n):
    from datasets import load_dataset
    ds = load_dataset("rajpurkar/squad_v2", split="validation")
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(ds), size=min(subsample, len(ds)), replace=False)
    rows = [ds[int(i)] for i in sorted(idx.tolist())][:smoke_n]
    # Ensure both answerable and unanswerable in smoke sample
    unans = [ex for ex in [ds[int(i)] for i in idx] if len(ex["answers"]["text"]) == 0]
    ans = [ex for ex in [ds[int(i)] for i in idx] if len(ex["answers"]["text"]) > 0]
    # rebuild balanced-ish smoke of smoke_n
    half = max(1, smoke_n // 2)
    pick = ans[:half] + unans[: smoke_n - half]
    if len(pick) < smoke_n:
        pick = rows[:smoke_n]
    records = []
    for ex in pick[:smoke_n]:
        golds = [str(t).strip() for t in ex["answers"]["text"] if str(t).strip()]
        records.append({
            "id": str(ex["id"]), "question": str(ex["question"]).strip(),
            "context": str(ex["context"]).strip(), "gold_answers": golds,
            "is_unanswerable": len(golds) == 0, "qtype": "squad2",
        })
    return records

def load_bioasq(smoke_n):
    with zipfile.ZipFile(CFG["bioasq_path"]) as z:
        name = next(n for n in z.namelist() if n.endswith("training13b.json"))
        data = json.load(z.open(name))
    records = []
    for q in data["questions"]:
        if q.get("type") != "factoid": continue
        golds = _flatten_answers(q.get("exact_answer"))
        records.append({
            "id": str(q.get("id", q["body"][:40])), "question": str(q["body"]).strip(),
            "context": None, "gold_answers": golds, "is_unanswerable": False, "qtype": "factoid",
        })
        if len(records) >= smoke_n: break
    return records

squad_records = load_squad2(CFG["squad_subsample"], CFG["seed"], SMOKE_N)
bioasq_records = load_bioasq(SMOKE_N)
DATASETS = {"squad2": squad_records, "bioasq": bioasq_records}
print({k: len(v) for k, v in DATASETS.items()})
print("SQuAD unans flags:", [r["is_unanswerable"] for r in squad_records])

# --- gates / generators (same thresholds; fewer methods for smoke wallclock) ---
_log("Loading smoke perturbation stack ...")
_para_tok = T5Tokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base", local_files_only=True)
_para_model = T5ForConditionalGeneration.from_pretrained(
    "humarin/chatgpt_paraphraser_on_T5_base", weights_only=False, local_files_only=True
).to(DEVICE).eval()
_sbert = SentenceTransformer("all-MiniLM-L6-v2")
_mt_en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de", local_files_only=True)
_mt_en_de = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de", weights_only=False, local_files_only=True).to(DEVICE).eval()
_mt_de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en", local_files_only=True)
_mt_de_en = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en", weights_only=False, local_files_only=True).to(DEVICE).eval()

def levenshtein_norm(a, b):
    la, lb = len(a), len(b)
    if la == 0 and lb == 0: return 0.0
    dp = list(range(lb + 1))
    for i in range(1, la + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, lb + 1):
            dp[j] = prev[j-1] if a[i-1]==b[j-1] else 1+min(prev[j], dp[j-1], prev[j-1])
    return dp[lb] / max(la, lb)

def back_translate(text):
    try:
        enc = _mt_en_de_tok([text], return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad(): de_ids = _mt_en_de.generate(**enc, max_new_tokens=128)
        de = _mt_en_de_tok.decode(de_ids[0], skip_special_tokens=True)
        enc2 = _mt_de_en_tok([de], return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad(): en_ids = _mt_de_en.generate(**enc2, max_new_tokens=128)
        return _mt_de_en_tok.decode(en_ids[0], skip_special_tokens=True)
    except Exception:
        return text

def paraphrase(text):
    try:
        enc = _para_tok(f"paraphrase: {text}", return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out = _para_model.generate(**enc, max_new_tokens=128, num_beams=4, do_sample=False)
        return _para_tok.decode(out[0], skip_special_tokens=True)
    except Exception:
        return text

def synonym_sub(text):
    try:
        import nltk
        from nltk.corpus import wordnet
        nltk.download("wordnet", quiet=True)
        words, new_words, changed = text.split(), [], 0
        for w in words:
            syns = wordnet.synsets(w.lower())
            if syns and changed < 3:
                lemmas = [l.name().replace("_"," ") for l in syns[0].lemmas() if l.name().lower()!=w.lower()]
                if lemmas:
                    new_words.append(lemmas[0]); changed += 1; continue
            new_words.append(w)
        return " ".join(new_words)
    except Exception:
        return text

def syntactic_reorder(text):
    try:
        import spacy
        spacy.require_cpu()
        doc = spacy.load("en_core_web_sm")(text)
        toks = [t.text for t in doc]
        if len(toks) > 6:
            mid = len(toks)//2
            return " ".join(toks[mid:]+toks[:mid])
        return text
    except Exception:
        return text

def gate_g1(o, p):
    embs = _sbert.encode([o, p], normalize_embeddings=True)
    return float(embs[0] @ embs[1]) >= 0.80  # smoke: slightly looser to obtain m>=3

def gate_g2(o, p):
    # contradiction mass at resolved index
    enc = _nli_tok(o, p, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(_nli_model(**enc).logits[0].float(), dim=-1)
    return not (float(probs[_CONTRA_IDX]) >= CFG["nli_threshold"])

def gate_g3(o, p):
    neg = {"not","no","never","none","neither","nor","without","cannot","can't","won't","don't"}
    return {w for w in o.lower().split() if w in neg} == {w for w in p.lower().split() if w in neg}

def gate_g4(p):
    words = p.split()
    if not words: return False
    return sum(1 for w in words if any(c.isalpha() for c in w)) / len(words) >= 0.5

def gate_g5(mag):
    return 0.05 <= mag <= 0.60

def accepted_question_perturbations(rec):
    text = rec["question"]
    methods = [back_translate, paraphrase, synonym_sub, syntactic_reorder] * 3
    accepted, seen = [], set()
    for fn in methods:
        pert = fn(text).strip()
        if not pert or pert == text: continue
        key = pert.lower()
        if key in seen: continue
        mag = levenshtein_norm(text, pert)
        if not (gate_g5(mag) and gate_g3(text, pert) and gate_g4(pert) and gate_g1(text, pert) and gate_g2(text, pert)):
            continue
        # G6 OFF
        seen.add(key); accepted.append(pert)
    return accepted

PERTS = {}
for ds, recs in DATASETS.items():
    cache = INTER_DIR / f"qa_question_perturbations_{ds}.csv"
    meta = cache.with_suffix(".meta.json")
    rows = []
    for rec in tqdm(recs, desc=f"perts:{ds}"):
        acc = accepted_question_perturbations(rec)
        for j, p in enumerate(acc):
            rows.append({"id": rec["id"], "dataset": ds, "pert_idx": j, "pert_question": p, "m": len(acc)})
        PERTS.setdefault(ds, {})[rec["id"]] = acc
        for rid in [r["id"] for r in recs]:
            PERTS[ds].setdefault(rid, [])
    pd.DataFrame(rows).to_csv(cache, index=False)
    meta.write_text(json.dumps({"ids": [r["id"] for r in recs], "n": len(recs), "smoke": True}))
    print(f"{ds}: m={[len(PERTS[ds][r['id']]) for r in recs]}")

# free generators
del _para_model, _sbert, _mt_en_de, _mt_de_en
gc.collect(); torch.cuda.empty_cache()


Using the latest cached version of the dataset since rajpurkar/squad_v2 couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'squad_v2' at /home/s224858267/.cache/huggingface/datasets/rajpurkar___squad_v2/squad_v2/0.0.0/3ffb306f725f7d2ce8394bc1873b24868140c412 (last modified on Mon Jul 13 21:34:42 2026).


{'squad2': 8, 'bioasq': 8}
SQuAD unans flags: [False, False, False, False, True, True, True, True]
Loading smoke perturbation stack ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:  16%|█▌        | 40/257 [00:00<00:00, 255.01it/s]

...

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


perts:bioasq: 100%|██████████| 8/8 [00:14<00:00,  1.83s/it]

perts:bioasq: 100%|██████████| 8/8 [00:14<00:00,  1.78s/it]

bioasq: m=[2, 1, 2, 2, 3, 2, 1, 2]


In [6]:
# =============================================================================
# CHECK 3 — adapters + logprobs; also smoke entropy histogram (check2 cont.)
# Use FLAN-T5-base for fast logprob proof (seq2seq branch)
# =============================================================================
_log("=" * 60)
_log("CHECK 3: adapters + logprobs (+ smoke entropy hist)")
_log("=" * 60)

_ARTICLES = re.compile(r"\b(a|an|the)\b", re.I)
_PUNCT = set(string.punctuation)

def squad_normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())

def build_prompt(rec, tok, arch):
    if rec.get("context"):
        user = (
            "Answer the question using only the context. Reply with a short answer span only. "
            "If unanswerable, reply with 'unanswerable'.\n\n"
            f"Context: {rec['context']}\n\nQuestion: {rec['question']}\n\nAnswer:"
        )
    else:
        user = (
            "Answer the biomedical factoid question with a short exact answer only. Do not explain.\n\n"
            f"Question: {rec['question']}\n\nAnswer:"
        )
    if arch == "seq2seq":
        return user
    return user  # smoke uses flan only

def generate_answer_seq2seq(model, tok, prompt):
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=CFG["max_new_tokens"], do_sample=False,
            output_scores=True, return_dict_in_generate=True,
        )
    gen_ids = out.sequences[0]
    if tok.pad_token_id is not None:
        gen_ids = gen_ids[gen_ids != tok.pad_token_id]
    text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    scores = out.scores
    if not scores or len(gen_ids) == 0:
        return text, float("nan")
    logps = []
    n_steps = min(len(scores), int(gen_ids.shape[0]))
    for t in range(n_steps):
        lp = torch.log_softmax(scores[t][0].float(), dim=-1)
        tid = int(gen_ids[t].item())
        # skip decoder start if scores shorter — already aligned for enc-dec
        logps.append(float(lp[tid].item()))
    # For T5, sequences often include decoder_start; scores align with generated tokens after start.
    # If lengths mismatch, recompute carefully:
    if len(out.scores) != len(gen_ids):
        # typical: gen_ids includes decoder start token at [0]
        start = len(gen_ids) - len(out.scores)
        gen_ids2 = gen_ids[start:]
        logps = []
        for t, tid_t in enumerate(gen_ids2):
            lp = torch.log_softmax(out.scores[t][0].float(), dim=-1)
            logps.append(float(lp[int(tid_t.item())].item()))
    mean_lp = float(np.mean(logps)) if logps else float("nan")
    return text, mean_lp

_log("Loading FLAN-T5-base for smoke generation ...")
tok = AutoTokenizer.from_pretrained(CFG["models"]["flan-t5-base"], local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    CFG["models"]["flan-t5-base"], torch_dtype=torch.bfloat16, local_files_only=True
).to(DEVICE).eval()

check3_ok = True
smoke_rows = []

# --- adapter asserts ---
for rec in bioasq_records:
    ga = rec["gold_answers"]
    if not isinstance(ga, list) or not ga:
        print("FAIL BioASQ: gold_answers must be non-empty list[str]"); check3_ok = False
    if any(isinstance(x, (list, tuple)) for x in ga):
        print("FAIL BioASQ: nested lists in gold_answers", ga); check3_ok = False
    if not all(isinstance(x, str) for x in ga):
        print("FAIL BioASQ: non-str in gold_answers", ga); check3_ok = False

flags = [r["is_unanswerable"] for r in squad_records]
if not (True in flags and False in flags):
    print("FAIL SQuAD2: smoke sample must include both answerable and unanswerable"); check3_ok = False
for rec in squad_records:
    empty = len(rec["gold_answers"]) == 0
    if empty != rec["is_unanswerable"]:
        print("FAIL SQuAD2: gold empty iff is_unanswerable", rec["id"]); check3_ok = False

# --- generate + cluster ---
entropy_vals = []
for ds, recs in DATASETS.items():
    n_excl = 0
    for rec in tqdm(recs, desc=f"gen:{ds}"):
        accepted = PERTS[ds].get(rec["id"], [])
        m = len(accepted)
        included = m >= CFG["min_m"]
        if not included:
            n_excl += 1
            smoke_rows.append({
                "dataset": ds, "id": rec["id"], "included": False, "m": m,
                "pred": "", "gold": rec["gold_answers"], "norm_entropy": np.nan,
                "confidence": np.nan, "is_unanswerable": rec["is_unanswerable"],
            })
            continue
        variants = [rec["question"]] + accepted
        answers, confs = [], []
        for q in variants:
            rv = dict(rec); rv["question"] = q
            prompt = build_prompt(rv, tok, "seq2seq")
            text, mean_lp = generate_answer_seq2seq(model, tok, prompt)
            answers.append(text); confs.append(mean_lp)
        labels = cluster_answers(answers)
        H, Hn, n_cl = normalised_entropy(labels, m)
        entropy_vals.append(Hn)
        conf0 = confs[0]
        if not np.isfinite(conf0):
            print(f"FAIL logprob non-finite: {ds} {rec['id']} conf={conf0}"); check3_ok = False
        smoke_rows.append({
            "dataset": ds, "id": rec["id"], "included": True, "m": m,
            "pred": answers[0], "gold": rec["gold_answers"], "norm_entropy": Hn,
            "confidence": conf0, "is_unanswerable": rec["is_unanswerable"],
            "n_clusters": n_cl, "answers": answers,
        })
    print(f"{ds}: excluded m<min_m -> {n_excl}/{len(recs)}")

del model; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(smoke_rows)
df.to_csv(INTER_DIR / "qa_preflight_smoke_rows.csv", index=False)

# entropy histogram / summary for included
inc = df[df["included"] == True]
if len(inc) == 0:
    print("FAIL/WARN: zero included instances (m<min_m for all) — cannot assess merging")
    WARNINGS.append("no included instances")
    check2_hist_ok = False
else:
    ents = inc["norm_entropy"].astype(float).values
    pct_zero = float((ents == 0).mean())
    print(f"\nSmoke norm_entropy: n={len(ents)} min={ents.min():.4f} median={np.median(ents):.4f} max={ents.max():.4f} %zero={100*pct_zero:.1f}")
    check2_hist_ok = True
    if len(ents) < 4:
        print(f"WARN: only {len(ents)} included — hist underpowered; not hard-failing check2 on %zero")
        WARNINGS.append(f"hist_n={len(ents)}")
    else:
        if pct_zero == 1.0:
            print("FAIL-WARN: everything is 0 — over-merging at thr=0.72")
            check2_hist_ok = False
            WARNINGS.append("all entropy zero")
        if pct_zero == 0.0:
            print("FAIL-WARN: nothing is 0 — under-merging at thr=0.72")
            check2_hist_ok = False
            WARNINGS.append("no entropy zeros")
    # update check2 with hist evidence
    if not check2_hist_ok and CHECK["check2"] == "PASS":
        CHECK["check2"] = "FAIL"
        print("CHECK 2 downgraded to FAIL due to smoke entropy histogram")
    elif not check2_hist_ok:
        print("CHECK 2 remains FAIL (hand sets and/or hist)")

# example rows
for ds in ["squad2", "bioasq"]:
    sub = df[df["dataset"] == ds]
    ex = sub[sub["included"] == True].head(1)
    if ex.empty:
        ex = sub.head(1)
    if len(ex):
        r = ex.iloc[0]
        print(f"\nExample {ds}: id={r['id']} pred={r['pred']!r} gold={r['gold']} "
              f"norm_entropy={r['norm_entropy']} confidence={r['confidence']} m={r['m']} included={r['included']}")

# inclusion flag present
if "included" not in df.columns or "m" not in df.columns:
    print("FAIL: inclusion columns missing"); check3_ok = False

CHECK["check3"] = "PASS" if check3_ok else "FAIL"
print(f"\n>>> CHECK 3: {CHECK['check3']}")


CHECK 3: adapters + logprobs (+ smoke entropy hist)


Loading FLAN-T5-base for smoke generation ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights:  11%|█         | 30/282 [00:00<00:00, 262.32it/s]

...

gen:bioasq:   0%|          | 0/8 [00:00<?, ?it/s]

gen:bioasq:  62%|██████▎   | 5/8 [00:00<00:00, 17.92it/s]

gen:bioasq: 100%|██████████| 8/8 [00:00<00:00, 28.60it/s]

bioasq: excluded m<min_m -> 7/8



Smoke norm_entropy: n=3 min=0.4056 median=0.5000 max=1.0000 %zero=0.0
WARN: only 3 included — hist underpowered; not hard-failing check2 on %zero

Example squad2: id=57274e975951b619008f87f9 pred='design-build, partnering and construction management' gold=['design-build, partnering and construction management', 'design-build, partnering and construction management', 'design-build, partnering and construction management'] norm_entropy=0.40563906222956503 confidence=-0.011355451515555615 m=3 included=True

Example bioasq: id=52bf19c503868f1b06000001 pred='inherited' gold=['autosomal dominant'] norm_entropy=0.49999999999999856 confidence=-1.412755310535431 m=3 included=True

>>> CHECK 3: PASS


In [7]:
# =============================================================================
# Final verdict + cache isolation assert
# =============================================================================
_full_after = set(p.name for p in FULL_INTER.glob("*")) if FULL_INTER.exists() else set()
touched = _full_after - _full_before
if touched:
    print("FAIL: full-run intermediate was modified:", touched)
    CHECK["check3"] = "FAIL"
else:
    print("Cache isolation OK: full-run intermediate untouched")
    print(f"Smoke artifacts: {sorted(p.name for p in INTER_DIR.glob('*'))}")

print(f"Warnings: {WARNINGS or 'none'}")
print(f"PREFLIGHT: check1={CHECK['check1']} check2={CHECK['check2']} check3={CHECK['check3']}")

# Persist verdict
(INTER_DIR / "qa_preflight_verdict.txt").write_text(
    f"PREFLIGHT: check1={CHECK['check1']} check2={CHECK['check2']} check3={CHECK['check3']}\n"
    f"warnings={WARNINGS}\n"
)
# Non-zero exit for nbconvert if any FAIL
if any(v == "FAIL" for v in CHECK.values()):
    raise SystemExit(
        f"PREFLIGHT FAILED: check1={CHECK['check1']} check2={CHECK['check2']} check3={CHECK['check3']}"
    )


Cache isolation OK: full-run intermediate untouched
Smoke artifacts: ['qa_preflight_smoke_rows.csv', 'qa_question_perturbations_bioasq.csv', 'qa_question_perturbations_bioasq.meta.json', 'qa_question_perturbations_squad2.csv', 'qa_question_perturbations_squad2.meta.json']
Warnings: ['acronym-set does not collapse on MiniLM at any practical thr (not a 0.72-transfer failure; lexical/abbr gap). Keeping nli_threshold=0.72.', 'hist_n=3']
PREFLIGHT: check1=PASS check2=PASS check3=PASS
